UPLOAD DATASET

In [50]:
# Upload Dataset
from google.colab import files
uploaded = files.upload()

Saving 15939911.ann to 15939911 (1).ann
Saving 15939911.txt to 15939911 (1).txt
Saving 16778410.ann to 16778410 (1).ann
Saving 16778410.txt to 16778410 (1).txt
Saving 17803823.ann to 17803823 (1).ann
Saving 17803823.txt to 17803823 (1).txt
Saving 18236639.ann to 18236639 (1).ann
Saving 18236639.txt to 18236639 (1).txt
Saving 18258107.ann to 18258107 (1).ann
Saving 18258107.txt to 18258107 (1).txt
Saving 18416479.ann to 18416479 (1).ann
Saving 18416479.txt to 18416479 (1).txt
Saving 18561524.ann to 18561524 (1).ann
Saving 18561524.txt to 18561524 (1).txt
Saving 18666334.ann to 18666334 (1).ann
Saving 18666334.txt to 18666334 (1).txt
Saving 18787726.ann to 18787726 (1).ann
Saving 18787726.txt to 18787726 (1).txt
Saving 18815636.ann to 18815636 (1).ann
Saving 18815636.txt to 18815636 (1).txt
Saving 19009665.ann to 19009665 (1).ann
Saving 19009665.txt to 19009665 (1).txt
Saving 19214295.ann to 19214295 (1).ann
Saving 19214295.txt to 19214295 (1).txt
Saving 19307547.ann to 19307547 (1).ann


In [51]:
# Install & Import

import json
import re
import random
from collections import Counter

Tokenization + BIO Conversion

In [52]:
def tokenize_with_offsets(text):
    return [(m.group(), m.start(), m.end())
            for m in re.finditer(r'\w+|\S', text)]


def convert_to_bio(text, entities):
    tokens_with_offsets = tokenize_with_offsets(text)
    tags = ["O"] * len(tokens_with_offsets)

    for ent in entities:
        ent_start, ent_end, label = ent["start"], ent["end"], ent["label"]

        entity_token_indices = []
        for i, (token_text, token_start, token_end) in enumerate(tokens_with_offsets):
            # Check for overlap: token_start < ent_end AND token_end > ent_start
            # This condition means the token's span intersects with the entity's span.
            if token_start < ent_end and token_end > ent_start:
                entity_token_indices.append(i)

        if not entity_token_indices:
            continue

        # Mark the first token of the entity as B-label
        tags[entity_token_indices[0]] = f"B-{label}"

        # Mark subsequent tokens of the entity as I-label
        for i in range(1, len(entity_token_indices)):
            tags[entity_token_indices[i]] = f"I-{label}"

    return [(tokens_with_offsets[i][0], tags[i]) for i in range(len(tokens_with_offsets))]


Load & Convert Dataset


In [53]:
def load_and_convert(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    sentences = []
    for item in data:
        bio = convert_to_bio(item["text"], item["entities"])
        sentences.append(bio)

    return sentences


def save_bio(sentences, filename):
    with open(filename, "w") as f:
        for sent in sentences:
            for token, tag in sent:
                f.write(f"{token} {tag}\n")
            f.write("\n")

Split Dataset

In [54]:
def split_data(sentences, train=0.7, val=0.1, test=0.2, seed=42):
    random.seed(seed)
    random.shuffle(sentences)

    n = len(sentences)
    t = int(n * train)
    v = int(n * val)

    return (
        sentences[:t],
        sentences[t:t+v],
        sentences[t+v:]
    )

Load BIO into Tokens

In [55]:
def load_bio(filepath):
    sentences, tags = [], []
    s, t = [], []

    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                if s:
                    sentences.append(s)
                    tags.append(t)
                    s, t = [], []
            else:
                tok, tag = line.split()
                s.append(tok)
                t.append(tag)

    if s:
        sentences.append(s)
        tags.append(t)

    return sentences, tags

Preprocessing

In [56]:
def clean_token(token):
    token = token.lower()
    token = re.sub(r'\d', '0', token)
    return token


def preprocess(sentences):
    return [[clean_token(w) for w in sent] for sent in sentences]

Build Vocabulary

In [57]:
def build_vocab(sentences, tags):
    word_counter = Counter(w for s in sentences for w in s)
    tag_set = set(t for seq in tags for t in seq)

    word2idx = {"<PAD>": 0, "<UNK>": 1}
    for w in word_counter:
        word2idx[w] = len(word2idx)

    tag2idx = {"<PAD>": 0}
    for t in sorted(tag_set):
        tag2idx[t] = len(tag2idx)

    char2idx = {"<PAD>": 0, "<UNK>": 1}
    for w in word_counter:
        for ch in w:
            if ch not in char2idx:
                char2idx[ch] = len(char2idx)

    return word2idx, tag2idx, char2idx

Encoding

In [58]:
def encode(sentences, tags, word2idx, tag2idx, char2idx):
    X, y, X_char = [], [], []

    for sent, tag_seq in zip(sentences, tags):
        word_ids = [word2idx.get(w, 1) for w in sent]
        tag_ids = [tag2idx[t] for t in tag_seq]

        char_ids = []
        for w in sent:
            char_ids.append([char2idx.get(c, 1) for c in w])

        X.append(word_ids)
        y.append(tag_ids)
        X_char.append(char_ids)

    return X, y, X_char

Padding

In [59]:
def pad(seq, max_len, pad_val=0):
    return [s[:max_len] + [pad_val]*(max_len-len(s)) for s in seq]


def pad_chars(seq, max_len, max_word_len):
    out = []
    for sent in seq:
        s = []
        for w in sent:
            w = w[:max_word_len] + [0]*(max_word_len-len(w))
            s.append(w)
        while len(s) < max_len:
            s.append([0]*max_word_len)
        out.append(s[:max_len])
    return out

Mask

In [60]:
def create_mask(X):
    return [[1 if tok != 0 else 0 for tok in seq] for seq in X]

Run Full Pipeline

In [61]:
import json
import re

# Function to parse .ann file content
def parse_ann_content(ann_content):
    entities = []
    for line in ann_content.splitlines():
        if line.startswith('T'):  # Text-bound annotation
            parts = line.split('\t')
            if len(parts) >= 2:
                type_and_offsets = parts[1]

                # Split off the label first
                space_idx = type_and_offsets.find(' ')
                if space_idx == -1: # Malformed line, no space after label
                    print(f"Warning: Malformed annotation info (no space after label) in line: {line}")
                    continue

                label = type_and_offsets[:space_idx]
                offsets_str = type_and_offsets[space_idx+1:]

                # Split by semicolon for multiple spans
                span_definitions = offsets_str.split(';')

                for span_def in span_definitions:
                    # Each span_def should be "START END"
                    span_parts = span_def.strip().split(' ')
                    span_parts = [p for p in span_parts if p] # Remove empty strings from multiple spaces

                    if len(span_parts) == 2:
                        try:
                            start = int(span_parts[0])
                            end = int(span_parts[1])
                            entities.append({"start": start, "end": end, "label": label})
                        except ValueError:
                            print(f"Warning: Could not convert start/end to int in span '{span_def}' in line: {line}")
                    else:
                        print(f"Warning: Malformed span definition '{span_def}' in line: {line}")
    return entities

# Generate your_dataset.json from uploaded .txt and .ann files
json_data = []
txt_files = {name: content for name, content in uploaded.items() if name.endswith('.txt')}
ann_files = {name: content for name, content in uploaded.items() if name.endswith('.ann')}

for txt_filename, txt_content_bytes in txt_files.items():
    base_name = txt_filename.replace('.txt', '')
    ann_filename = base_name + '.ann'

    if ann_filename in ann_files:
        text = txt_content_bytes.decode('utf-8')
        ann_content = ann_files[ann_filename].decode('utf-8')
        entities = parse_ann_content(ann_content)
        json_data.append({"text": text, "entities": entities})
    else:
        print(f"Warning: No matching .ann file found for {txt_filename}")

# Save the combined data to your_dataset.json
with open("your_dataset.json", "w") as f:
    json.dump(json_data, f, indent=4)

print("Generated your_dataset.json from uploaded .txt and .ann files.")

# Step 1: Convert JSON → BIO
data = load_and_convert("your_dataset.json")
save_bio(data, "bio.txt")

# Step 2: Split
train, val, test = split_data(data)
save_bio(train, "train.txt")
save_bio(val, "val.txt")
save_bio(test, "test.txt")

# Step 3: Load train
train_sents, train_tags = load_bio("train.txt")

# Step 4: Preprocess
train_sents = preprocess(train_sents)

# Step 5: Vocabulary
word2idx, tag2idx, char2idx = build_vocab(train_sents, train_tags)

# Step 6: Encoding
X, y, X_char = encode(train_sents, train_tags, word2idx, tag2idx, char2idx)

# Step 7: Padding
MAX_LEN = max(len(s) for s in X)
MAX_WORD_LEN = max(len(w) for s in X_char for w in s)

X = pad(X, MAX_LEN)
y = pad(y, MAX_LEN)
X_char = pad_chars(X_char, MAX_LEN, MAX_WORD_LEN)

# Step 8: Mask
mask = create_mask(X)

print("✅ Pipeline Completed")
print("Sentences:", len(X))
print("Max Length:", MAX_LEN)
print("Word Vocab:", len(word2idx))
print("Tag Vocab:", len(tag2idx))

Generated your_dataset.json from uploaded .txt and .ann files.
✅ Pipeline Completed
Sentences: 140
Max Length: 1213
Word Vocab: 7138
Tag Vocab: 83


In [62]:
# Shape Consistency Check
len(X) == len(y)

True

In [63]:
# Tag Integrity Check

idx2tag = {v: k for k, v in tag2idx.items()}

for seq in y:
    for i, tag_id in enumerate(seq):
        tag = idx2tag[tag_id]
        # Handle '<PAD>' tags which should not be checked for BIO integrity
        if tag == '<PAD>':
            continue

        if tag.startswith("I"):
            # If it's an 'I' tag at the beginning of a sequence or preceded by 'O'
            # this is an invalid BIO sequence.
            if i == 0 or idx2tag[seq[i-1]] == "O":
                print(f"Invalid BIO sequence found: 'I' tag '{tag}' at index {i} following '{idx2tag[seq[i-1]]}' (or at start of sequence).")


In [64]:
# Vocabulary Coverage

unknown_ratio = sum(1 for sent in X for w in sent if w == 1) / sum(len(s) for s in X)
print("UNK ratio:", unknown_ratio)

UNK ratio: 0.0


In [65]:
# Padding & Mask Check

print(mask[0])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [66]:
# Label Distribution

from collections import Counter
print(Counter(tag for seq in train_tags for tag in seq))

Counter({'O': 40352, 'I-Lab_value': 3394, 'B-Diagnostic_procedure': 3264, 'I-Diagnostic_procedure': 3141, 'B-Sign_symptom': 2361, 'I-Detailed_description': 2231, 'B-Biological_structure': 2078, 'B-Detailed_description': 2064, 'B-Lab_value': 1997, 'I-Biological_structure': 1783, 'I-Sign_symptom': 1088, 'I-Date': 1058, 'I-History': 1057, 'B-Disease_disorder': 958, 'I-Dosage': 857, 'B-Medication': 816, 'B-Therapeutic_procedure': 718, 'I-Disease_disorder': 628, 'I-Age': 539, 'B-Date': 539, 'I-Therapeutic_procedure': 428, 'B-Clinical_event': 422, 'I-Duration': 363, 'I-Medication': 345, 'I-Other_entity': 311, 'B-Dosage': 298, 'B-History': 267, 'B-Severity': 260, 'I-Family_history': 250, 'B-Nonbiological_location': 241, 'I-Nonbiological_location': 238, 'B-Duration': 204, 'B-Coreference': 200, 'I-Distance': 199, 'I-Area': 178, 'I-Clinical_event': 161, 'B-Age': 143, 'I-Volume': 140, 'B-Sex': 135, 'B-Administration': 135, 'I-Frequency': 93, 'B-Distance': 86, 'I-Time': 83, 'I-Coreference': 74, 'B

Word Embedding Layer (PyTorch)

In [67]:
import torch
import torch.nn as nn

class WordEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

    def forward(self, x):
        return self.embedding(x)

In [68]:
#Usage

vocab_size = len(word2idx)
embed_dim = 100

word_embed = WordEmbedding(vocab_size, embed_dim)

sample = torch.tensor(X[:2])  # batch of sentences
out = word_embed(sample)

print(out.shape)
# (batch_size, seq_len, embed_dim)

torch.Size([2, 1213, 100])
